# SMART Data Preparation (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-baseline/notebooks/smart_data_prep_colab.ipynb)

This notebook owns the **data artifact stage** only:
- stage raw shards from GCS,
- preprocess selected SMART splits,
- persist processed data to Drive,
- write a reusable data-prep artifact.


In [7]:
# Step 1: Sync this repo and bootstrap deterministic Colab runtime
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


[repo] existing checkout: /content/wosac-sim-agents-experiments
[drive] /content/drive already mounted
[drive] Verified read/write access: /content/drive/MyDrive/wosac_experiments
[setup] cache hit -> skipping deterministic setup: ok numpy=2.2.6 pandas=2.2.3
[setup] cached ready state at /content/.wosac_setup_cache.json
repo_rev: 54a5627


In [ ]:
# # Suggested defaults: persist validation first, then training later if needed.
# %env WOSAC_RUN_MODE=full
# %env SMART_BASELINE_PROFILE=paper_repro
# %env SMART_STAGE_TRAINING=0
# %env SMART_STAGE_VALIDATION=1
# %env SMART_PREPROCESS_SPLITS=validation
# %env SMART_PERSIST_SPLITS=validation
# %env SMART_PERSIST_PROCESSED_ROOT=/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
# %env SMART_RESUMABLE_PREPROCESS=1
# %env SMART_PREPROCESS_MAX_SHARDS_PER_RUN=0


In [3]:
%env WOSAC_RUN_MODE=full
%env SMART_BASELINE_PROFILE=paper_repro
%env SMART_STAGE_TRAINING=1
%env SMART_STAGE_VALIDATION=0
%env SMART_PREPROCESS_SPLITS=training
%env SMART_PERSIST_SPLITS=training
%env SMART_PERSIST_PROCESSED_ROOT=/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
%env SMART_GCS_TRAIN_SHARDS=64
%env SMART_RUN_PREPROCESS=1
%env SMART_BIND_PERSISTED_ROOT=0
%env SMART_RESUMABLE_PREPROCESS=1
%env SMART_PREPROCESS_MAX_SHARDS_PER_RUN=8


env: WOSAC_RUN_MODE=full
env: SMART_BASELINE_PROFILE=paper_repro
env: SMART_STAGE_TRAINING=1
env: SMART_STAGE_VALIDATION=0
env: SMART_PREPROCESS_SPLITS=training
env: SMART_PERSIST_SPLITS=training
env: SMART_PERSIST_PROCESSED_ROOT=/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
env: SMART_GCS_TRAIN_SHARDS=64
env: SMART_RUN_PREPROCESS=1
env: SMART_BIND_PERSISTED_ROOT=0
env: SMART_RESUMABLE_PREPROCESS=1
env: SMART_PREPROCESS_MAX_SHARDS_PER_RUN=8


In [4]:
# Step 2: Load config, select splits, stage raw data from GCS, and inspect processed roots
import hashlib
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config


def _bool_env(name: str, default: bool = False) -> bool:
    text = os.environ.get(name, '').strip().lower()
    if text in {'1', 'true', 'yes', 'y', 'on'}:
        return True
    if text in {'0', 'false', 'no', 'n', 'off'}:
        return False
    return bool(default)


def _csv_env(name: str, default: list[str]) -> list[str]:
    text = os.environ.get(name, '').strip()
    if not text:
        return list(default)
    out = []
    for part in text.split(','):
        part = part.strip().lower()
        if part and part not in out:
            out.append(part)
    return out or list(default)


def _ensure_colab_auth() -> None:
    if 'google.colab' not in sys.modules:
        return
    try:
        from google.colab import auth  # type: ignore
        auth.authenticate_user()
        print('colab_gcs_auth: OK')
    except Exception as exc:
        print(f'colab_gcs_auth: skipped ({exc})')


def _gcloud_ls(pattern: str) -> list[str]:
    try:
        out = subprocess.check_output(
            ['bash', '-lc', f"gcloud storage ls '{pattern}'"],
            text=True,
            stderr=subprocess.STDOUT,
        )
        return [line.strip() for line in out.splitlines() if line.strip()]
    except Exception:
        return []


def _choose_shards(pattern: str, limit: int) -> list[str]:
    rows = sorted(_gcloud_ls(pattern))
    return rows[:limit] if int(limit) > 0 else rows


def _local_shards(split_dir: Path) -> list[Path]:
    return sorted([p for p in split_dir.glob('*.tfrecord*') if p.is_file()])


def _count_processed(split_dir: Path) -> int:
    files = [p for p in split_dir.glob('*.pkl') if p.is_file()]
    files += [p for p in split_dir.glob('*.pickle') if p.is_file()]
    return len(files)


def _stage_split(*, gcs_root: str, split_name: str, shard_limit: int, local_split_dir: Path, force: bool) -> dict:
    local_split_dir.mkdir(parents=True, exist_ok=True)
    pattern = f'{gcs_root}/{split_name}/{split_name}.tfrecord-*'
    remote_shards = _choose_shards(pattern, shard_limit)
    existing = {p.name for p in _local_shards(local_split_dir)}
    pending = remote_shards if force else [u for u in remote_shards if Path(u).name not in existing]

    copied = 0
    errors: list[str] = []
    for uri in pending:
        try:
            subprocess.run(['gcloud', 'storage', 'cp', '--no-clobber', uri, str(local_split_dir) + '/'], check=True)
            copied += 1
        except Exception as exc:
            errors.append(f'{uri}: {type(exc).__name__}')

    local_files = _local_shards(local_split_dir)
    return {
        'split': split_name,
        'pattern': pattern,
        'requested_limit': int(shard_limit),
        'remote_candidates': len(remote_shards),
        'pending_before_copy': len(pending),
        'copied': int(copied),
        'local_count': len(local_files),
        'local_sample': [str(p) for p in local_files[:5]],
        'errors': errors,
    }


EXPERIMENT_SLUG = 'smart-baseline'
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root='.')
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root='.')
display(bundle.summary_table)

RUN = dict(cfg.get('run', {}))
SMART = dict(cfg.get('smart', {}))
SMART_PROFILES = dict(SMART.get('profiles', {}))

RUN_NAME = str(RUN.get('run_name', 'dev'))
RUN_PREFIX = str(RUN.get('run_prefix', 'smart_baseline'))
PERSIST_ROOT = str(RUN.get('persist_root', '/content/drive/MyDrive/wosac_experiments'))
RUN_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode('utf-8')).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f'{RUN_PREFIX}_{RUN_NAME}'
persist_run_dir.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT_DIR = persist_run_dir / 'outputs' / RUN_TAG
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = os.environ.get('WOSAC_RUN_MODE', 'full').strip().lower() or 'full'
if RUN_MODE not in {'pilot', 'full'}:
    print(f"Unknown WOSAC_RUN_MODE={RUN_MODE!r}; falling back to 'full'.")
    RUN_MODE = 'full'

SMART_PROFILE = os.environ.get('SMART_BASELINE_PROFILE', 'paper_repro').strip() or 'paper_repro'
SMART_EFFECTIVE = dict(SMART)
if SMART_PROFILE in SMART_PROFILES:
    SMART_EFFECTIVE.update(dict(SMART_PROFILES[SMART_PROFILE]))

raw_root_override = os.environ.get('SMART_RAW_DATA_ROOT', '').strip()
processed_root_override = os.environ.get('SMART_PROCESSED_DATA_ROOT', '').strip()
if raw_root_override:
    SMART_EFFECTIVE['raw_data_root'] = raw_root_override
if processed_root_override:
    SMART_EFFECTIVE['processed_data_root'] = processed_root_override

SMART_TRAIN_SEED = int(os.environ.get('SMART_TRAIN_SEED', str(SMART_EFFECTIVE.get('seed', 2))))
DATA_SOURCE_MODE = os.environ.get('SMART_DATA_SOURCE', 'gcs_stage').strip().lower() or 'gcs_stage'
RUN_DATA_STAGE = _bool_env('SMART_RUN_DATA_STAGE', True)
FORCE_DATA_REDOWNLOAD = _bool_env('SMART_FORCE_DATA_REDOWNLOAD', False)
GCS_DATASET_ROOT = os.environ.get('SMART_GCS_DATASET_ROOT', 'gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario').strip().rstrip('/')
TRAIN_SPLIT = os.environ.get('SMART_GCS_TRAIN_SPLIT', 'training').strip() or 'training'
VAL_SPLIT = os.environ.get('SMART_GCS_VAL_SPLIT', 'validation').strip() or 'validation'
TRAIN_SHARD_LIMIT = int(os.environ.get('SMART_GCS_TRAIN_SHARDS', '8' if RUN_MODE == 'pilot' else '64'))
VAL_SHARD_LIMIT = int(os.environ.get('SMART_GCS_VAL_SHARDS', '2' if RUN_MODE == 'pilot' else '8'))

STAGE_TRAINING = _bool_env('SMART_STAGE_TRAINING', False)
STAGE_VALIDATION = _bool_env('SMART_STAGE_VALIDATION', True)
SELECTED_STAGE_SPLITS = []
if STAGE_TRAINING:
    SELECTED_STAGE_SPLITS.append('training')
if STAGE_VALIDATION:
    SELECTED_STAGE_SPLITS.append('validation')
if not SELECTED_STAGE_SPLITS:
    raise RuntimeError('Select at least one split via SMART_STAGE_TRAINING=1 or SMART_STAGE_VALIDATION=1.')

PREPROCESS_SPLITS = _csv_env('SMART_PREPROCESS_SPLITS', SELECTED_STAGE_SPLITS)
PERSIST_SPLITS = _csv_env('SMART_PERSIST_SPLITS', PREPROCESS_SPLITS)
PERSIST_PROCESSED_ROOT = Path(os.environ.get('SMART_PERSIST_PROCESSED_ROOT', '/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed')).expanduser()
RESUMABLE_PREPROCESS = _bool_env('SMART_RESUMABLE_PREPROCESS', True)
PREPROCESS_MAX_SHARDS_PER_RUN = int(os.environ.get('SMART_PREPROCESS_MAX_SHARDS_PER_RUN', '0'))
if RESUMABLE_PREPROCESS and not processed_root_override:
    SMART_EFFECTIVE['processed_data_root'] = str(PERSIST_PROCESSED_ROOT)

RAW_ROOT = Path(str(SMART_EFFECTIVE.get('raw_data_root', '/content/SMART/data/waymo/scenario'))).expanduser()
PROCESSED_ROOT = Path(str(SMART_EFFECTIVE.get('processed_data_root', '/content/SMART/data/waymo_processed'))).expanduser()
PREPROCESS_STATE_ROOT = PROCESSED_ROOT / '.preprocess_state'
RAW_DIRS = {'training': RAW_ROOT / 'training', 'validation': RAW_ROOT / 'validation'}
PROCESSED_DIRS = {'training': PROCESSED_ROOT / 'training', 'validation': PROCESSED_ROOT / 'validation'}
for path in [*RAW_DIRS.values(), *PROCESSED_DIRS.values()]:
    path.mkdir(parents=True, exist_ok=True)


data_stage_manifest = {
    'data_source_mode': DATA_SOURCE_MODE,
    'run_data_stage': bool(RUN_DATA_STAGE),
    'force_data_redownload': bool(FORCE_DATA_REDOWNLOAD),
    'gcs_dataset_root': GCS_DATASET_ROOT,
    'selected_stage_splits': SELECTED_STAGE_SPLITS,
    'train_shard_limit': int(TRAIN_SHARD_LIMIT),
    'val_shard_limit': int(VAL_SHARD_LIMIT),
    'resumable_preprocess': bool(RESUMABLE_PREPROCESS),
    'preprocess_max_shards_per_run': int(PREPROCESS_MAX_SHARDS_PER_RUN),
    'results': {},
}
if DATA_SOURCE_MODE == 'gcs_stage' and RUN_DATA_STAGE:
    _ensure_colab_auth()
    if 'training' in SELECTED_STAGE_SPLITS:
        data_stage_manifest['results']['training'] = _stage_split(
            gcs_root=GCS_DATASET_ROOT,
            split_name=TRAIN_SPLIT,
            shard_limit=TRAIN_SHARD_LIMIT,
            local_split_dir=RAW_DIRS['training'],
            force=FORCE_DATA_REDOWNLOAD,
        )
    if 'validation' in SELECTED_STAGE_SPLITS:
        data_stage_manifest['results']['validation'] = _stage_split(
            gcs_root=GCS_DATASET_ROOT,
            split_name=VAL_SPLIT,
            shard_limit=VAL_SHARD_LIMIT,
            local_split_dir=RAW_DIRS['validation'],
            force=FORCE_DATA_REDOWNLOAD,
        )
else:
    data_stage_manifest['results']['note'] = 'staging_skipped'

raw_counts = {split: len(_local_shards(path)) for split, path in RAW_DIRS.items()}
processed_counts = {split: _count_processed(path) for split, path in PROCESSED_DIRS.items()}
preprocess_policy = {
    'selected_stage_splits': SELECTED_STAGE_SPLITS,
    'preprocess_splits': PREPROCESS_SPLITS,
    'persist_splits': PERSIST_SPLITS,
    'raw_counts': raw_counts,
    'processed_counts': processed_counts,
    'resumable_preprocess': bool(RESUMABLE_PREPROCESS),
    'preprocess_max_shards_per_run': int(PREPROCESS_MAX_SHARDS_PER_RUN),
    'preprocess_state_root': str(PREPROCESS_STATE_ROOT),
}

print('RUN_TAG:', RUN_TAG)
print('RUN_MODE:', RUN_MODE)
print('SMART_PROFILE:', SMART_PROFILE)
print('SMART train seed:', SMART_TRAIN_SEED)
print('SMART raw_data_root:', SMART_EFFECTIVE.get('raw_data_root', ''))
print('SMART processed_data_root:', SMART_EFFECTIVE.get('processed_data_root', ''))
print('PERSIST_PROCESSED_ROOT:', PERSIST_PROCESSED_ROOT)
print('RESUMABLE_PREPROCESS:', RESUMABLE_PREPROCESS)
print('PREPROCESS_MAX_SHARDS_PER_RUN:', PREPROCESS_MAX_SHARDS_PER_RUN)
print('PREPROCESS_STATE_ROOT:', PREPROCESS_STATE_ROOT)
print('cfg_hash:', cfg_hash)
print('data_stage_manifest:')
print(json.dumps(data_stage_manifest, indent=2, sort_keys=True))
print('preprocess_policy:')
print(json.dumps(preprocess_policy, indent=2, sort_keys=True))


,field,value
0,slug,smart-baseline
1,title,SMART Baseline Wrapper
2,objective,Reproduce SMART baseline with thin wrapper flo...
3,notebook,experiments/smart-baseline/notebooks/smart-bas...
4,workflow,src/workflows/smart_baseline_flow.py
5,config_file,/content/wosac-sim-agents-experiments/configs/...


colab_gcs_auth: OK
RUN_TAG: 20260406T123518Z
RUN_MODE: full
SMART_PROFILE: paper_repro
SMART train seed: 2
SMART raw_data_root: /content/SMART/data/waymo/scenario
SMART processed_data_root: /content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
PERSIST_PROCESSED_ROOT: /content/drive/MyDrive/wosac_experiments/datasets/waymo_processed
RESUMABLE_PREPROCESS: True
PREPROCESS_MAX_SHARDS_PER_RUN: 8
PREPROCESS_STATE_ROOT: /content/drive/MyDrive/wosac_experiments/datasets/waymo_processed/.preprocess_state
cfg_hash: 3a55f8daa9f07df7b3389f7127e4ee88bab0f11075d1ac8e306b7777e205820f
data_stage_manifest:
{
  "data_source_mode": "gcs_stage",
  "force_data_redownload": false,
  "gcs_dataset_root": "gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario",
  "preprocess_max_shards_per_run": 8,
  "results": {
    "training": {
      "copied": 64,
      "errors": [],
      "local_count": 64,
      "local_sample": [
        "/content/SMART/data/waymo/scenario/training/training.tfrecord-000

In [5]:
# Step 3: Build preprocess command plan
from src.workflows import run_smart_baseline_flow

BASELINE_CKPT_DIR = str(persist_run_dir / 'checkpoints' / f'smart_baseline_{SMART_PROFILE}')
flow_bundle = run_smart_baseline_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root='.',
    resume_from_existing=True,
    smart_repo_url=str(SMART_EFFECTIVE.get('repo_url', 'https://github.com/rainmaker22/SMART.git')),
    smart_repo_branch=str(SMART_EFFECTIVE.get('branch', 'main')),
    smart_repo_commit=str(SMART_EFFECTIVE.get('repo_commit', '')).strip(),
    smart_repo_dir=str(SMART_EFFECTIVE.get('repo_dir', '/content/SMART')),
    smart_train_config=str(SMART_EFFECTIVE.get('train_config', 'configs/train/train_scalable.yaml')),
    smart_val_config=str(SMART_EFFECTIVE.get('val_config', 'configs/validation/validation_scalable.yaml')),
    smart_ckpt_path='',
    smart_save_ckpt_path=BASELINE_CKPT_DIR,
    smart_raw_data_root=str(SMART_EFFECTIVE.get('raw_data_root', '/content/SMART/data/waymo/scenario')),
    smart_processed_data_root=str(PROCESSED_ROOT),
    smart_install_pyg=bool(SMART_EFFECTIVE.get('install_pyg', True)),
    smart_preprocess_max_shards=PREPROCESS_MAX_SHARDS_PER_RUN,
    smart_env_lockfile=str(SMART_EFFECTIVE.get('env_lockfile', '')).strip(),
    smart_train_seed=SMART_TRAIN_SEED,
    smart_deterministic_train=bool(SMART_EFFECTIVE.get('deterministic_train', True)),
    smart_train_launcher_path=str(SMART_EFFECTIVE.get('train_launcher_path', 'scripts/smart_train_repro.py')),
    smart_profile=SMART_PROFILE,
    smart_force_preprocess=True,
    sync_smart_repo=True,
)

print('flow_summary:')
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))
if flow_bundle.summary.get('status') == 'sync_failed':
    raise RuntimeError(f"SMART repo sync failed before preprocess: {flow_bundle.summary.get('sync_error', 'unknown')}")
print('preprocess_train_cmd:')
print(flow_bundle.command_plan['preprocess_train_cmd'])
print('preprocess_val_cmd:')
print(flow_bundle.command_plan['preprocess_val_cmd'])


flow_summary:
{
  "checkpoint_manifest": {
    "checkpoints": [],
    "pretrain_ckpt_path": "",
    "pretrain_ckpt_sha256": "",
    "resume": {
      "candidate_sample": [],
      "checkpoint_exists": false,
      "checkpoint_path": "",
      "num_candidates": 0,
      "source": "none"
    },
    "save_ckpt_exists": false,
    "save_ckpt_path": "/content/drive/MyDrive/wosac_experiments/smart_baseline_dev/checkpoints/smart_baseline_paper_repro"
  },
  "config_hash": "47f2f7eb0a36d0bfa6267ee89198c043a54dbd131cd28a142c0dca0dcad9144b",
  "created_utc": "2026-04-06T12:40:32Z",
  "data_manifest": {
    "processed_data_root": "/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed",
    "raw_data_root": "/content/SMART/data/waymo/scenario",
    "splits": {
      "training": {
        "processed": {
          "exists": true,
          "listing_sha256": "",
          "num_files": 0,
          "path": "/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed/training",
      

In [8]:
# Step 4: Optional preprocess execution (selected splits only)
import subprocess

RUN_PREPROCESS = _bool_env('SMART_RUN_PREPROCESS', True)
SELECTED_PREPROCESS_SPLITS = [s for s in PREPROCESS_SPLITS if s in {'training', 'validation'}]
if not SELECTED_PREPROCESS_SPLITS:
    raise RuntimeError('SMART_PREPROCESS_SPLITS resolved to no valid splits.')


def _run_cmd(cmd: str, idx: int, total: int, *, label: str) -> None:
    print(f'[{label} {idx}/{total}] {cmd}')
    merged_cmd = f'export PYTHONUNBUFFERED=1; {cmd}'
    process = subprocess.Popen(
        ['bash', '-lc', merged_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    recent_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        recent_lines.append(line)
        if len(recent_lines) > 80:
            recent_lines.pop(0)
    return_code = int(process.wait())
    if return_code == 0:
        return
    diag = {
        'failed_command': cmd,
        'exit_code': return_code,
        'selected_preprocess_splits': SELECTED_PREPROCESS_SPLITS,
        'raw_counts': preprocess_policy.get('raw_counts', {}),
        'processed_counts': preprocess_policy.get('processed_counts', {}),
        'recent_output': ''.join(recent_lines[-20:]),
    }
    print('step4_diagnostics:')
    print(json.dumps(diag, indent=2, sort_keys=True))
    raise subprocess.CalledProcessError(return_code, ['bash', '-lc', merged_cmd])

cmd_map = {
    'training': flow_bundle.command_plan['preprocess_train_cmd'],
    'validation': flow_bundle.command_plan['preprocess_val_cmd'],
}
if RUN_PREPROCESS:
    for split in SELECTED_PREPROCESS_SPLITS:
        if int(preprocess_policy['raw_counts'].get(split, 0)) <= 0:
            raise RuntimeError(f'No raw shards available for {split}. Rerun Step 2 with staging enabled.')
    for idx, split in enumerate(SELECTED_PREPROCESS_SPLITS, start=1):
        _run_cmd(cmd_map[split], idx, len(SELECTED_PREPROCESS_SPLITS), label=f'preprocess:{split}')
else:
    print('SMART_RUN_PREPROCESS=0 -> skipping preprocess execution.')

print('Data preprocess execution block done.')


[preprocess:training 1/1] cd /content/wosac-sim-agents-experiments && python /content/wosac-sim-agents-experiments/scripts/ensure_smart_preprocess_runtime.py && python /content/wosac-sim-agents-experiments/scripts/smart_preprocess_resumable.py --input_dir /content/SMART/data/waymo/scenario/training --output_dir /content/drive/MyDrive/wosac_experiments/datasets/waymo_processed/training --split training --smart-repo-dir /content/SMART --skip-existing --max-shards 8
[smart-preprocess-setup] runtime ready
{
  "input_dir": "/content/SMART/data/waymo/scenario/training",
  "log_every": 25,
  "max_shards": 8,
  "output_dir": "/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed/training",
  "progress_json": "",
  "skip_existing": true,
  "smart_repo_dir": "/content/SMART",
  "split": "training",
  "state_dir": ""
}
[smart-preprocess] processing shard 1/8: training.tfrecord-00000-of-01000
W0000 00:00:1775479699.412313   13770 gpu_bfc_allocator.cc:47] Overriding orig_value setting b

In [ ]:
# from pathlib import Path

# src = Path("/content/SMART/data/waymo_processed/validation")
# dst = Path("/content/drive/MyDrive/wosac_experiments/datasets/waymo_processed/validation")

# def count_files(path: Path):
#     return len([p for p in path.rglob("*") if p.is_file()])

# print("src_exists:", src.exists(), "src_files:", count_files(src) if src.exists() else 0)
# print("dst_exists:", dst.exists(), "dst_files:", count_files(dst) if dst.exists() else 0)


In [ ]:
# %env SMART_BIND_PERSISTED_ROOT=0

In [9]:
# Step 5: Persist selected processed splits to Drive and optionally bind the canonical SMART path to the Drive copy
import subprocess
from pathlib import Path

PERSIST_SPLITS_EFFECTIVE = [s for s in PERSIST_SPLITS if s in {'training', 'validation'}]
PERSIST_MANIFESTS_DIR = RUN_OUTPUT_DIR / 'persist_manifests'
PERSIST_MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
BIND_AFTER_PERSIST = _bool_env('SMART_BIND_PERSISTED_ROOT', False)
SMART_CANONICAL_PROCESSED_ROOT = Path('/content/SMART/data/waymo_processed')
persist_rows = []

for split in PERSIST_SPLITS_EFFECTIVE:
    manifest_path = PERSIST_MANIFESTS_DIR / f'{split}.json'
    cmd = [
        'python',
        f'{REPO_DIR}/scripts/persist_processed_split.py',
        '--src-root', str(PROCESSED_ROOT),
        '--dst-root', str(PERSIST_PROCESSED_ROOT),
        '--split', split,
        '--manifest-json', str(manifest_path),
    ]
    print('[persist]', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    persist_rows.append({'split': split, 'manifest_json': str(manifest_path)})

if BIND_AFTER_PERSIST:
    target = PERSIST_PROCESSED_ROOT.resolve()
    canonical = SMART_CANONICAL_PROCESSED_ROOT
    canonical.parent.mkdir(parents=True, exist_ok=True)
    if canonical.is_symlink():
        if canonical.resolve() != target:
            canonical.unlink()
            canonical.symlink_to(target, target_is_directory=True)
    elif canonical.exists():
        try:
            has_entries = any(canonical.iterdir())
        except Exception:
            has_entries = True
        if has_entries and canonical.resolve() != target:
            raise RuntimeError(
                f'Cannot bind {canonical} to {target}: canonical path already exists and is non-empty. '
                'Move or remove it first, or disable SMART_BIND_PERSISTED_ROOT.'
            )
        if not has_entries:
            canonical.rmdir()
            canonical.symlink_to(target, target_is_directory=True)
    else:
        canonical.symlink_to(target, target_is_directory=True)
    print('bound_canonical_processed_root:', canonical, '->', target)

print('persist_rows:', json.dumps(persist_rows, indent=2, sort_keys=True))


[persist] python /content/wosac-sim-agents-experiments/scripts/persist_processed_split.py --src-root /content/drive/MyDrive/wosac_experiments/datasets/waymo_processed --dst-root /content/drive/MyDrive/wosac_experiments/datasets/waymo_processed --split training --manifest-json /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260406T123518Z/persist_manifests/training.json
persist_rows: [
  {
    "manifest_json": "/content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260406T123518Z/persist_manifests/training.json",
    "split": "training"
  }
]


In [10]:
# Step 6: Write data-prep artifact (Drive)
from src.workflows import write_json


def _count_processed(split_dir: Path) -> int:
    files = [p for p in split_dir.glob('*.pkl') if p.is_file()]
    files += [p for p in split_dir.glob('*.pickle') if p.is_file()]
    return len(files)

local_processed_after = {split: _count_processed(PROCESSED_DIRS[split]) for split in PROCESSED_DIRS}
drive_processed_after = {
    split: _count_processed((PERSIST_PROCESSED_ROOT / split))
    for split in ['training', 'validation']
    if (PERSIST_PROCESSED_ROOT / split).exists()
}

payload = {
    'run_id': 'smart_data_prep_v0',
    'run_tag': RUN_TAG,
    'run_mode': RUN_MODE,
    'smart_profile': SMART_PROFILE,
    'status': 'completed',
    'run_output_dir': str(RUN_OUTPUT_DIR),
    'raw_data_root': str(RAW_ROOT),
    'processed_data_root': str(PROCESSED_ROOT),
    'persist_processed_root': str(PERSIST_PROCESSED_ROOT),
    'resumable_preprocess': bool(RESUMABLE_PREPROCESS),
    'preprocess_max_shards_per_run': int(PREPROCESS_MAX_SHARDS_PER_RUN),
    'preprocess_state_root': str(PREPROCESS_STATE_ROOT),
    'selected_stage_splits': SELECTED_STAGE_SPLITS,
    'preprocess_splits': SELECTED_PREPROCESS_SPLITS,
    'persist_splits': PERSIST_SPLITS_EFFECTIVE,
    'data_stage_manifest': data_stage_manifest,
    'flow_summary': flow_bundle.summary,
    'local_processed_counts_after': local_processed_after,
    'drive_processed_counts_after': drive_processed_after,
    'persist_manifests_dir': str(PERSIST_MANIFESTS_DIR),
    'provenance': {
        'config_hash': cfg_hash,
        'created_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'experiment_slug': EXPERIMENT_SLUG,
        'source_notebook': 'smart_data_prep_colab.ipynb',
    },
    'notes': [
        'This notebook owns preprocessing and persistence only.',
        'Future train/eval sessions should set SMART_PROCESSED_DATA_ROOT to the persisted root or rely on the canonical symlink.',
    ],
}

drive_path = RUN_OUTPUT_DIR / 'smart_data_prep_v0.json'
write_json(drive_path, payload)
run_manifest_path = RUN_OUTPUT_DIR / 'run_manifest.json'
write_json(run_manifest_path, {
    'run_id': 'smart_data_prep_v0',
    'run_tag': RUN_TAG,
    'created_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'experiment_slug': EXPERIMENT_SLUG,
    'run_name': RUN_NAME,
    'run_prefix': RUN_PREFIX,
    'persist_root': PERSIST_ROOT,
    'config_hash': cfg_hash,
    'raw_data_root': str(RAW_ROOT),
    'processed_data_root': str(PROCESSED_ROOT),
    'persist_processed_root': str(PERSIST_PROCESSED_ROOT),
    'resumable_preprocess': bool(RESUMABLE_PREPROCESS),
    'preprocess_max_shards_per_run': int(PREPROCESS_MAX_SHARDS_PER_RUN),
    'preprocess_state_root': str(PREPROCESS_STATE_ROOT),
    'selected_stage_splits': SELECTED_STAGE_SPLITS,
    'preprocess_splits': SELECTED_PREPROCESS_SPLITS,
    'persist_splits': PERSIST_SPLITS_EFFECTIVE,
})

print('drive_path:', drive_path)
print('run_manifest_path:', run_manifest_path)
print('local_processed_counts_after:', local_processed_after)
print('drive_processed_counts_after:', drive_processed_after)


drive_path: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260406T123518Z/smart_data_prep_v0.json
run_manifest_path: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260406T123518Z/run_manifest.json
local_processed_counts_after: {'training': 3948, 'validation': 2361}
drive_processed_counts_after: {'training': 3948, 'validation': 2361}
